In [ ]:
import pathlib
import os

import pandas as pd

import plotly
import plotly.graph_objects as go
import plotly.express as px

import trapallada.xsa.access
import trapallada.xsa.util
import trapallada.plot

So that the notebook is aware of modifications in the the `import`ed modules

In [ ]:
%load_ext autoreload
%autoreload 2

So that *plotly* pictures can be exported to HTML

In [ ]:
plotly.offline.init_notebook_mode()

Path to the raw data

In [ ]:
data_dir = pathlib.Path(os.environ['MY_DATA_DIR']) / 'trapallada'
display(data_dir)
assert data_dir.exists()

The signal of interest...

In [ ]:
signal = '0764191301/P0764191301PNS003SRCTSR8002'
# signal = '0203540301/P0203540301PNS003SRCTSR8003'
# signal = '0146390101/P0146390101PNS005SRCTSR8002'
tar_file, fit_file = signal.split('/')

...is read

In [ ]:
tar_file_df, tar_file_headers_df, tar_mask_df = trapallada.xsa.util.read_tar((data_dir / tar_file).with_suffix('.tar'))
# tar_file_df

A convenience function

In [ ]:
def plot_signal(series: pd.Series):
    
    fig = px.line(x=series.index,y=series.values)

    return fig

# Raw signal

In [ ]:
fig = plot_signal(tar_file_df[(fit_file + '.FTZ', 'RATE')])
fig

## Interpolated

In [ ]:
interpolated_df = tar_file_df.interpolate(method='linear', axis=0)
interpolated_fig = plot_signal(interpolated_df[(fit_file + '.FTZ', 'RATE')])
interpolated_fig

The two signals can be (plotted and) explored *in sync*...but the resulting figure is not exportable.

In [ ]:
fig_widget = go.FigureWidget(fig)
interpolated_fig_widget = go.FigureWidget(interpolated_fig)

def sync_axes(source_figure, target_figure):
    def update_layout(trace, points, state):
        target_figure.layout.update(source_figure.layout)
    return update_layout

fig_widget.layout.on_change(sync_axes(fig_widget, interpolated_fig_widget), 'xaxis.range', 'yaxis.range')
interpolated_fig_widget.layout.on_change(sync_axes(interpolated_fig_widget, fig_widget), 'xaxis.range', 'yaxis.range')

# display(fig_widget)
# display(interpolated_fig_widget)

# Binned signal

In [ ]:
bin_size = 63

In [ ]:
fig = plot_signal(
    trapallada.xsa.util.bin_fits_df(
        tar_file_df,
        bin_size=bin_size
    )[(fit_file + '.FTZ', 'RATE')]
)
fig

## Interpolated

In [ ]:
interpolated_df = trapallada.xsa.util.bin_fits_df(
    tar_file_df.interpolate(method='linear', axis=0),
    bin_size=bin_size
)
interpolated_fig = plot_signal(interpolated_df[(fit_file + '.FTZ', 'RATE')])
interpolated_fig